# EDA и эксперименты: риск срыва доставки заказа

Ноутбук показывает базовую разведку синтетического датасета и связывает наблюдения с обучением моделей.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.config import load_train_config
from src.data_generation import save_dataset

config = load_train_config(PROJECT_ROOT / 'configs' / 'train_config.yaml')
dataset_path = PROJECT_ROOT / config['data']['raw_dataset_path']
save_dataset(dataset_path, rows=config['data']['rows'], random_seed=config['project']['random_seed'])
df = pd.read_csv(dataset_path)
df.head()

In [ ]:
df.shape, df['is_late_delivery'].value_counts(normalize=True).round(4)

In [ ]:
df[[
    'order_value',
    'delivery_distance_km',
    'warehouse_backlog',
    'courier_experience_months',
    'route_complexity',
    'historical_late_rate',
]].describe().round(2)

In [ ]:
df.groupby('weather_condition')['is_late_delivery'].mean().sort_values(ascending=False).round(3)

In [ ]:
df.groupby('traffic_level')['is_late_delivery'].mean().sort_values(ascending=False).round(3)

In [ ]:
df.groupby('delivery_type')['is_late_delivery'].mean().sort_values(ascending=False).round(3)

Ожидаемые закономерности видны: срочные доставки, плохая погода, сильный трафик и сложные маршруты повышают риск задержки.

In [ ]:
from src.train import main as train_main

train_main()
pd.read_json(PROJECT_ROOT / 'artifacts' / 'leaderboard.json').T